# Stage 4 (Bonus): Drift Detection

DVC stage `drift`. The data spans July 2015 – August 2017, so it can be treated as a stream over time.

- **Reference window**: bookings arriving in `reference_years` (what the model would originally be trained on)
- **Current window**: bookings arriving in `current_years` (new, incoming data)

Three checks are made:

1. **Data drift** — Population Stability Index (PSI) per feature, plus a Kolmogorov–Smirnov test for numeric features. PSI < 0.1 is stable, 0.1–0.2 is moderate, > 0.2 is significant drift.
2. **Target drift** — change in the cancellation rate between windows.
3. **Concept drift** — a model trained only on the reference window is scored on a reference hold-out and on the current window. A large F1 drop means the relationship between features and cancellations has changed.

**Response**: if too many features drift or F1 drops more than `max_f1_drop`, the report sets `retrain_recommended: true`, meaning the model should be retrained on recent data (update the data and run `dvc repro`).

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import ks_2samp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

## Load parameters and split data into time windows

In [ ]:
with open('params.yaml') as f:
    params = yaml.safe_load(f)

seed = params['base']['random_state']
target = params['base']['target']
drift_cfg = params['drift']
train_cfg = params['train']

processed_dir = Path(params['data']['processed_dir'])
data = pd.concat([pd.read_csv(processed_dir / 'train.csv'), pd.read_csv(processed_dir / 'test.csv')],
                 ignore_index=True)

reference = data[data['arrival_date_year'].isin(drift_cfg['reference_years'])].reset_index(drop=True)
current = data[data['arrival_date_year'].isin(drift_cfg['current_years'])].reset_index(drop=True)
features = [c for c in data.columns if c != target and c not in drift_cfg['exclude_columns']]
print(f'Reference rows: {len(reference)}, current rows: {len(current)}, features checked: {len(features)}')

## Data drift: PSI and KS test

In [ ]:
def psi(ref, cur, bins):
    if ref.nunique() <= bins:
        cats = np.union1d(ref.unique(), cur.unique())
        ref_pct = ref.value_counts(normalize=True).reindex(cats, fill_value=0).values
        cur_pct = cur.value_counts(normalize=True).reindex(cats, fill_value=0).values
    else:
        edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        ref_pct = np.histogram(ref, edges)[0] / len(ref)
        cur_pct = np.histogram(cur, edges)[0] / len(cur)
    ref_pct = np.clip(ref_pct, 1e-4, None)
    cur_pct = np.clip(cur_pct, 1e-4, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


rows = []
for col in features:
    value = psi(reference[col], current[col], drift_cfg['psi_bins'])
    ks_p = float(ks_2samp(reference[col], current[col]).pvalue) if reference[col].nunique() > drift_cfg['psi_bins'] else None
    rows.append({'feature': col, 'psi': round(value, 4), 'ks_pvalue': ks_p,
                 'drifted': value > drift_cfg['psi_threshold']})

psi_df = pd.DataFrame(rows).sort_values('psi', ascending=False).reset_index(drop=True)
drifted_features = psi_df.loc[psi_df['drifted'], 'feature'].tolist()
drift_share = len(drifted_features) / len(features)
print(f'Drifted features ({len(drifted_features)}/{len(features)}): {drifted_features}')
psi_df.head(10)

## Target drift

In [ ]:
ref_rate = float(reference[target].mean())
cur_rate = float(current[target].mean())
print(f'Cancellation rate  reference: {ref_rate:.3f}  current: {cur_rate:.3f}  change: {cur_rate - ref_rate:+.3f}')

## Concept drift: performance of a reference-only model on new data

In [ ]:
ref_train, ref_hold = train_test_split(reference, test_size=0.2, random_state=seed, stratify=reference[target])
all_features = [c for c in data.columns if c != target]

ref_model = RandomForestClassifier(
    n_estimators=train_cfg['n_estimators'],
    max_depth=train_cfg['max_depth'],
    min_samples_split=train_cfg['min_samples_split'],
    min_samples_leaf=train_cfg['min_samples_leaf'],
    class_weight=train_cfg['class_weight'],
    n_jobs=-1,
    random_state=seed,
)
ref_model.fit(ref_train[all_features], ref_train[target])

f1_reference = float(f1_score(ref_hold[target], ref_model.predict(ref_hold[all_features])))
f1_current = float(f1_score(current[target], ref_model.predict(current[all_features])))
f1_drop = f1_reference - f1_current
print(f'F1 on reference hold-out: {f1_reference:.4f}')
print(f'F1 on current window:     {f1_current:.4f}')
print(f'F1 drop:                  {f1_drop:+.4f}')

## Drift decision and report

In [ ]:
data_drift = drift_share > drift_cfg['drifted_share_threshold']
concept_drift = f1_drop > drift_cfg['max_f1_drop']
retrain = bool(data_drift or concept_drift)

report = {
    'reference_rows': int(len(reference)),
    'current_rows': int(len(current)),
    'features_checked': len(features),
    'drifted_features': len(drifted_features),
    'drift_share': round(drift_share, 4),
    'max_psi': float(psi_df['psi'].max()),
    'reference_cancel_rate': round(ref_rate, 4),
    'current_cancel_rate': round(cur_rate, 4),
    'f1_reference': round(f1_reference, 4),
    'f1_current': round(f1_current, 4),
    'f1_drop': round(f1_drop, 4),
    'data_drift_detected': bool(data_drift),
    'concept_drift_detected': bool(concept_drift),
    'retrain_recommended': retrain,
}

Path('metrics').mkdir(exist_ok=True)
with open('metrics/drift_report.json', 'w') as f:
    json.dump(report, f, indent=2)
psi_df.to_csv('metrics/drift_features.csv', index=False)
print(json.dumps(report, indent=2))

In [ ]:
top = psi_df.head(15)
colors = ['crimson' if d else 'steelblue' for d in top['drifted']][::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top['feature'][::-1], top['psi'][::-1], color=colors)
ax.axvline(drift_cfg['psi_threshold'], color='gray', linestyle='--', label=f"PSI threshold = {drift_cfg['psi_threshold']}")
ax.set_xlabel('PSI (reference vs current)')
ax.set_title('Feature Drift: Population Stability Index')
ax.legend()
fig.tight_layout()
Path('plots').mkdir(exist_ok=True)
fig.savefig('plots/drift_psi.png', dpi=120)
plt.close(fig)
print('Retrain recommended' if retrain else 'No retraining needed')